In [228]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer, MissingIndicator
import sqlite3
import json

## 📚 Part A: Conceptual Foundation


### Task 1 — Short Notes

**What is Data Analysis?**  
Data analysis means looking at raw data and trying to find useful information from it. We clean the data, explore it, and then use it to answer questions or make predictions.  
For example — if a bank wants to know which customers might not repay loans, they can analyze past data and find patterns.

**How to Plan a Data Science Project?**  
1. Understand the problem first — what are we trying to predict?  
2. Collect the data from different sources  
3. Clean the data — handle missing values, outliers etc  
4. Do feature engineering — create new useful columns  
5. Build the model  
6. Evaluate results  
In our case: predict if a customer will **default on a loan** (yes/no)

**How to Frame a Machine Learning Problem?**  
Our problem is a **CLASSIFICATION** problem.  
- Input (X) = customer features like age, income, credit score etc  
- Output (Y) = `default_flag` (0 = no default, 1 = default)  
- Algorithm type = **Supervised Learning** because we have labeled data  
- Metric = F1 Score or ROC-AUC (because data is imbalanced)

In [229]:
# A tensor is basically a container for numbers
# Different dimensions = different types of tensors

# 0D Tensor = Scalar (just one number)
scalar = np.array(42)
print("0D Tensor (Scalar):", scalar, "| Shape:", scalar.shape)

# 1D Tensor = Vector (a list of numbers)
vector = np.array([10, 20, 30, 40, 50])
print("1D Tensor (Vector):", vector, "| Shape:", vector.shape)

# 2D Tensor = Matrix (rows and columns - like a table)
matrix = np.array([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9]])
print("2D Tensor (Matrix):\n", matrix, "\nShape:", matrix.shape)

# 3D Tensor = Cube of numbers (like RGB image with 3 channels)
tensor_3d = np.array([[[1, 2], [3, 4]],
                       [[5, 6], [7, 8]]])
print("3D Tensor:\n", tensor_3d, "\nShape:", tensor_3d.shape)

# useful numpy operations
a = np.array([100, 200, 300, 400, 500])
print("\nArray:", a)
print("Mean:", np.mean(a), "| Sum:", np.sum(a))
print("Reshape to (5,1):\n", a.reshape(5, 1))

0D Tensor (Scalar): 42 | Shape: ()
1D Tensor (Vector): [10 20 30 40 50] | Shape: (5,)
2D Tensor (Matrix):
 [[1 2 3]
 [4 5 6]
 [7 8 9]] 
Shape: (3, 3)
3D Tensor:
 [[[1 2]
  [3 4]]

 [[5 6]
  [7 8]]] 
Shape: (2, 2, 2)

Array: [100 200 300 400 500]
Mean: 300.0 | Sum: 1500
Reshape to (5,1):
 [[100]
 [200]
 [300]
 [400]
 [500]]


## Part-B


In [230]:
# importing csv dataset

df_csv = pd.read_csv("transactions.csv")
print(df_csv.head())

  customer_id  loan_amount loan_purpose  transaction_count  spending_ratio
0   CUST00001     25700.13          Car                 26         19.0994
1   CUST00002     19264.58     Business                  2         27.1723
2   CUST00003     23983.44    Education                 28         41.9300
3   CUST00004     58439.10          Car                 48         44.8451
4   CUST00005     63903.19    Education                 10         46.2541


In [231]:
# importing json dataset

df_json = pd.read_json("customer_metadata.json")
print(df_json.head())

  customer_id   age  gender region education_level employment_type
0   CUST00001  59.0  Female  South        Graduate   Self-Employed
1   CUST00002  49.0  Female   West       Secondary   Self-Employed
2   CUST00003  35.0  Female   East        Graduate            None
3   CUST00004  63.0  Female   East        Graduate   Self-Employed
4   CUST00005  28.0  Female  South        Graduate            None


In [232]:
# importing sql dataset

conn = sqlite3.connect("loan_repayment.db")
df_sql = pd.read_sql_query("SELECT * FROM loan_repayment_history", conn)
print(df_sql.head())


  customer_id  annual_income  credit_score  repayment_history
0   CUST00001   38384.982865    565.520304                  3
1   CUST00002   54156.786444    580.911557                  2
2   CUST00003   88523.013804    621.473062                  1
3   CUST00004  139662.121852    620.076082                  1
4   CUST00005   58780.063998    533.745555                  2


In [233]:
# importing dataset from api in from of json

with open("economic_indicators_api.json", 'r') as f:
    api_data = json.load(f)
df_api = pd.json_normalize(api_data['records'])
print(f"API data Shape: {df_api.shape}")
df_api.head()

API data Shape: (1000, 3)


,customer_id,join_date,default_flag
0,CUST00001,2022-03-04,0
1,CUST00002,2016-04-01,0
2,CUST00003,2015-04-13,0
3,CUST00004,2018-01-31,0
4,CUST00005,2017-09-30,0


In [234]:
# merge all the datsets

df = df_csv.merge(df_json, on="customer_id").merge(df_sql, on="customer_id").merge(df_api, on="customer_id")
df.replace(["None", "none", "NULL", "null", "NaN", "nan", "N/A", "n/a", " ", "-"], np.nan, inplace=True)
df = df.where(df.notna(), other=np.nan)

display(df.head())
print(df.shape)

,customer_id,loan_amount,loan_purpose,transaction_count,spending_ratio,age,gender,region,education_level,employment_type,annual_income,credit_score,repayment_history,join_date,default_flag
0,CUST00001,25700.13,Car,26,19.0994,59.0,Female,South,Graduate,Self-Employed,38384.982865,565.520304,3,2022-03-04,0
1,CUST00002,19264.58,Business,2,27.1723,49.0,Female,West,Secondary,Self-Employed,54156.786444,580.911557,2,2016-04-01,0
2,CUST00003,23983.44,Education,28,41.9300,35.0,Female,East,Graduate,NaN,88523.013804,621.473062,1,2015-04-13,0
3,CUST00004,58439.10,Car,48,44.8451,63.0,Female,East,Graduate,Self-Employed,139662.121852,620.076082,1,2018-01-31,0
4,CUST00005,63903.19,Education,10,46.2541,28.0,Female,South,Graduate,NaN,58780.063998,533.745555,2,2017-09-30,0


(1000, 15)


## Part-C

In [235]:
# exploring the dataset

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        1000 non-null   object 
 1   loan_amount        1000 non-null   float64
 2   loan_purpose       1000 non-null   object 
 3   transaction_count  1000 non-null   int64  
 4   spending_ratio     1000 non-null   float64
 5   age                950 non-null    float64
 6   gender             960 non-null    object 
 7   region             1000 non-null   object 
 8   education_level    1000 non-null   object 
 9   employment_type    940 non-null    object 
 10  annual_income      950 non-null    float64
 11  credit_score       960 non-null    float64
 12  repayment_history  1000 non-null   int64  
 13  join_date          1000 non-null   object 
 14  default_flag       1000 non-null   int64  
dtypes: float64(5), int64(3), object(7)
memory usage: 117.3+ KB
None


In [236]:
print(df.describe())

         loan_amount  transaction_count  spending_ratio         age  \
count    1000.000000        1000.000000     1000.000000  950.000000   
mean    55609.235800          25.288000       28.928296   42.531579   
std     72016.026543          14.044868       15.871188   12.590350   
min      2375.180000           1.000000        1.238200   21.000000   
25%     20980.077500          13.000000       16.714375   31.250000   
50%     35477.625000          25.500000       27.333150   43.000000   
75%     62765.617500          37.000000       39.624425   53.000000   
max    907596.960000          49.000000       83.727900   64.000000   

       annual_income  credit_score  repayment_history  default_flag  
count   9.500000e+02    960.000000        1000.000000   1000.000000  
mean    1.490647e+05    599.094264           1.569000      0.007000  
std     2.124312e+05     85.412915           1.238052      0.083414  
min     1.569859e+04    290.000000           0.000000      0.000000  
25%     6.

In [237]:
print(df.isnull().sum())

customer_id           0
loan_amount           0
loan_purpose          0
transaction_count     0
spending_ratio        0
age                  50
gender               40
region                0
education_level       0
employment_type      60
annual_income        50
credit_score         40
repayment_history     0
join_date             0
default_flag          0
dtype: int64


In [238]:
# from ydata_profiling import ProfileReport

# profile = ProfileReport(
#     df,
#     title="Customer Credit Risk - Data Profiling Report",
#     explorative=True,   
#     minimal=False       
# )
# profile.to_file("data_profiling_report.html")

# print("Report saved! Open data_profiling_report.html in your browser.")

# profile.to_notebook_iframe()

In [239]:
# handling missing values with simple imputer

df_simple = df.copy()

mean_imputer = SimpleImputer(strategy="mean")
median_imputer = SimpleImputer(strategy="median")
frequent_imputer = SimpleImputer(strategy="most_frequent")

df_simple[["gender","employment_type"]] = frequent_imputer.fit_transform(df_simple[["gender","employment_type"]])
df_simple["age"] = mean_imputer.fit_transform(df_simple[["age"]])
df_simple[["credit_score","annual_income"]] = median_imputer.fit_transform(df_simple[["credit_score","annual_income"]])

print("after applying simple imputer")

print(df_simple.isnull().sum())
display(df_simple.head())



after applying simple imputer
customer_id          0
loan_amount          0
loan_purpose         0
transaction_count    0
spending_ratio       0
age                  0
gender               0
region               0
education_level      0
employment_type      0
annual_income        0
credit_score         0
repayment_history    0
join_date            0
default_flag         0
dtype: int64


,customer_id,loan_amount,loan_purpose,transaction_count,spending_ratio,age,gender,region,education_level,employment_type,annual_income,credit_score,repayment_history,join_date,default_flag
0,CUST00001,25700.13,Car,26,19.0994,59.0,Female,South,Graduate,Self-Employed,38384.982865,565.520304,3,2022-03-04,0
1,CUST00002,19264.58,Business,2,27.1723,49.0,Female,West,Secondary,Self-Employed,54156.786444,580.911557,2,2016-04-01,0
2,CUST00003,23983.44,Education,28,41.9300,35.0,Female,East,Graduate,Salaried,88523.013804,621.473062,1,2015-04-13,0
3,CUST00004,58439.10,Car,48,44.8451,63.0,Female,East,Graduate,Self-Employed,139662.121852,620.076082,1,2018-01-31,0
4,CUST00005,63903.19,Education,10,46.2541,28.0,Female,South,Graduate,Salaried,58780.063998,533.745555,2,2017-09-30,0


In [240]:
# missing indicator and random sample imputation
df_random = df.copy()

def random_sample_imputer(df, col, random_state=18):
    null_count = df[col].isnull().sum()

    if null_count > 0:
        random_value = df[col].dropna().sample(
            n=null_count,
            replace=True,
            random_state=random_state
        ).values

        df_ = df.copy()
        df_.loc[df_[col].isnull(), col] = random_value

        return df_
    return df


miss_ind = MissingIndicator(features='missing-only')
miss_ind_array = miss_ind.fit_transform(df)

miss_ind_col = [df.columns[col] + '_missing' for col in miss_ind.features_]
df_miss_ind = pd.DataFrame(miss_ind_array.astype(int), columns=miss_ind_col)

print("\nNew indicator columns created:", miss_ind_col)
display(df_miss_ind.head())

df_random = pd.concat([df_random.reset_index(drop=True), df_miss_ind], axis=1)

print("\nShape after adding indicator columns:", df_random.shape)

for col in df_random.columns:
    df_random = random_sample_imputer(df_random,col)
    
print("after applying random impuataion")    
display(df_random.head())
print(df_random.isnull().sum())



New indicator columns created: ['age_missing', 'gender_missing', 'employment_type_missing', 'annual_income_missing', 'credit_score_missing']


,age_missing,gender_missing,employment_type_missing,annual_income_missing,credit_score_missing
0,0,0,0,0,0
1,0,0,0,0,0
2,0,0,1,0,0
3,0,0,0,0,0
4,0,0,1,0,0



Shape after adding indicator columns: (1000, 20)
after applying random impuataion


,customer_id,loan_amount,loan_purpose,transaction_count,spending_ratio,age,gender,region,education_level,employment_type,annual_income,credit_score,repayment_history,join_date,default_flag,age_missing,gender_missing,employment_type_missing,annual_income_missing,credit_score_missing
0,CUST00001,25700.13,Car,26,19.0994,59.0,Female,South,Graduate,Self-Employed,38384.982865,565.520304,3,2022-03-04,0,0,0,0,0,0
1,CUST00002,19264.58,Business,2,27.1723,49.0,Female,West,Secondary,Self-Employed,54156.786444,580.911557,2,2016-04-01,0,0,0,0,0,0
2,CUST00003,23983.44,Education,28,41.9300,35.0,Female,East,Graduate,Salaried,88523.013804,621.473062,1,2015-04-13,0,0,0,1,0,0
3,CUST00004,58439.10,Car,48,44.8451,63.0,Female,East,Graduate,Self-Employed,139662.121852,620.076082,1,2018-01-31,0,0,0,0,0,0
4,CUST00005,63903.19,Education,10,46.2541,28.0,Female,South,Graduate,Salaried,58780.063998,533.745555,2,2017-09-30,0,0,0,1,0,0


customer_id                0
loan_amount                0
loan_purpose               0
transaction_count          0
spending_ratio             0
age                        0
gender                     0
region                     0
education_level            0
employment_type            0
annual_income              0
credit_score               0
repayment_history          0
join_date                  0
default_flag               0
age_missing                0
gender_missing             0
employment_type_missing    0
annual_income_missing      0
credit_score_missing       0
dtype: int64


In [241]:

# knn imputer

df_knn = df.copy()

df_knn.drop(['customer_id', 'default_flag', 'join_date'], axis=1, inplace=True)

cols = ['gender', 'region', 'employment_type', 'loan_purpose', 'education_level']
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan)
df_knn[cols] = oe.fit_transform(df_knn[cols])

knn = KNNImputer(n_neighbors=10, weights='distance')
df_knn = pd.DataFrame(knn.fit_transform(df_knn), columns=df_knn.columns)

df_knn[cols] = df_knn[cols].round().astype(int)
df_knn[cols] = oe.inverse_transform(df_knn[cols])

print('After applying KNNImputer:')
display(df_knn.head(10))
print(df_knn.isnull().sum())

After applying KNNImputer:


,loan_amount,loan_purpose,transaction_count,spending_ratio,age,gender,region,education_level,employment_type,annual_income,credit_score,repayment_history
0,25700.13,Car,26.0,19.0994,59.0,Female,South,Graduate,Self-Employed,38384.982865,565.520304,3.0
1,19264.58,Business,2.0,27.1723,49.0,Female,West,Secondary,Self-Employed,54156.786444,580.911557,2.0
2,23983.44,Education,28.0,41.9300,35.0,Female,East,Graduate,Self-Employed,88523.013804,621.473062,1.0
3,58439.10,Car,48.0,44.8451,63.0,Female,East,Graduate,Self-Employed,139662.121852,620.076082,1.0
4,63903.19,Education,10.0,46.2541,28.0,Female,South,Graduate,Self-Employed,58780.063998,533.745555,2.0
5,13086.02,Education,39.0,17.2556,41.0,Female,East,Secondary,Salaried,458418.079889,654.852511,1.0
6,17727.11,Home,37.0,17.6691,59.0,Female,North,Primary,Salaried,47573.714010,548.864517,0.0
7,20571.90,Car,39.0,31.2673,39.0,Male,East,Graduate,Unemployed,177943.212904,587.497893,0.0
8,34501.93,Car,21.0,42.1541,43.0,Male,South,Graduate,Self-Employed,276619.631043,654.941926,2.0
9,9447.24,Car,28.0,27.3012,31.0,Female,South,Secondary,Self-Employed,259255.545054,548.654288,0.0


loan_amount          0
loan_purpose         0
transaction_count    0
spending_ratio       0
age                  0
gender               0
region               0
education_level      0
employment_type      0
annual_income        0
credit_score         0
repayment_history    0
dtype: int64


In [242]:
# iterative imputer or mice

df_mice = df.copy()

df_mice_dropped = df_mice[['customer_id','default_flag','join_date']].copy()
df_mice.drop(['customer_id','default_flag','join_date'], axis=1, inplace=True)

cols = ['gender','region','employment_type','loan_purpose','education_level']
oe = OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=np.nan)
df_mice[cols] = oe.fit_transform(df_mice[cols])

mice = IterativeImputer(max_iter=175, random_state=18)
df_mice = pd.DataFrame(mice.fit_transform(df_mice), columns=df_mice.columns)

df_mice[cols] = df_mice[cols].round().astype(int)
df_mice[cols] = oe.inverse_transform(df_mice[cols])

df_mice = pd.concat([df_mice_dropped.reset_index(drop=True), df_mice.reset_index(drop=True)], axis=1)
print('After applying MICE Algorithm: ')
display(df_mice.head(10))
print(df_mice.isnull().sum())

After applying MICE Algorithm: 


,customer_id,default_flag,join_date,loan_amount,loan_purpose,transaction_count,spending_ratio,age,gender,region,education_level,employment_type,annual_income,credit_score,repayment_history
0,CUST00001,0,2022-03-04,25700.13,Car,26.0,19.0994,59.0,Female,South,Graduate,Self-Employed,38384.982865,565.520304,3.0
1,CUST00002,0,2016-04-01,19264.58,Business,2.0,27.1723,49.0,Female,West,Secondary,Self-Employed,54156.786444,580.911557,2.0
2,CUST00003,0,2015-04-13,23983.44,Education,28.0,41.9300,35.0,Female,East,Graduate,Self-Employed,88523.013804,621.473062,1.0
3,CUST00004,0,2018-01-31,58439.10,Car,48.0,44.8451,63.0,Female,East,Graduate,Self-Employed,139662.121852,620.076082,1.0
4,CUST00005,0,2017-09-30,63903.19,Education,10.0,46.2541,28.0,Female,South,Graduate,Self-Employed,58780.063998,533.745555,2.0
5,CUST00006,0,2017-07-03,13086.02,Education,39.0,17.2556,41.0,Female,East,Secondary,Salaried,458418.079889,654.852511,1.0
6,CUST00007,0,2016-07-25,17727.11,Home,37.0,17.6691,59.0,Female,North,Primary,Salaried,47573.714010,548.864517,0.0
7,CUST00008,0,2016-02-24,20571.90,Car,39.0,31.2673,39.0,Male,East,Graduate,Unemployed,177943.212904,587.497893,0.0
8,CUST00009,0,2022-08-03,34501.93,Car,21.0,42.1541,43.0,Male,South,Graduate,Self-Employed,276619.631043,654.941926,2.0
9,CUST00010,0,2021-02-11,9447.24,Car,28.0,27.3012,31.0,Female,South,Secondary,Self-Employed,259255.545054,548.654288,0.0


customer_id          0
default_flag         0
join_date            0
loan_amount          0
loan_purpose         0
transaction_count    0
spending_ratio       0
age                  0
gender               0
region               0
education_level      0
employment_type      0
annual_income        0
credit_score         0
repayment_history    0
dtype: int64


In [243]:
# complete case analysis

df_cca = df.copy()

print("Shape BEFORE dropping missing rows:", df_cca.shape)
print("Total missing values:\n", df_cca.isnull().sum())

df_cca = df_cca.dropna()

print("\nShape AFTER dropping missing rows:", df_cca.shape)
print("Total rows dropped:", df.shape[0] - df_cca.shape[0])
print("Total missing values after CCA:\n", df_cca.isnull().sum())

Shape BEFORE dropping missing rows: (1000, 15)
Total missing values:
 customer_id           0
loan_amount           0
loan_purpose          0
transaction_count     0
spending_ratio        0
age                  50
gender               40
region                0
education_level       0
employment_type      60
annual_income        50
credit_score         40
repayment_history     0
join_date             0
default_flag          0
dtype: int64

Shape AFTER dropping missing rows: (773, 15)
Total rows dropped: 227
Total missing values after CCA:
 customer_id          0
loan_amount          0
loan_purpose         0
transaction_count    0
spending_ratio       0
age                  0
gender               0
region               0
education_level      0
employment_type      0
annual_income        0
credit_score         0
repayment_history    0
join_date            0
default_flag         0
dtype: int64


##### This tells that it is better to use simple imputer, knn imputer, iterative imputer and random sample imputer than just dropping the rows.

## Part-D Outlier Handling

In [244]:
display(df_mice)

,customer_id,default_flag,join_date,loan_amount,loan_purpose,transaction_count,spending_ratio,age,gender,region,education_level,employment_type,annual_income,credit_score,repayment_history
0,CUST00001,0,2022-03-04,25700.13,Car,26.0,19.0994,59.000000,Female,South,Graduate,Self-Employed,38384.982865,565.520304,3.0
1,CUST00002,0,2016-04-01,19264.58,Business,2.0,27.1723,49.000000,Female,West,Secondary,Self-Employed,54156.786444,580.911557,2.0
2,CUST00003,0,2015-04-13,23983.44,Education,28.0,41.9300,35.000000,Female,East,Graduate,Self-Employed,88523.013804,621.473062,1.0
3,CUST00004,0,2018-01-31,58439.10,Car,48.0,44.8451,63.000000,Female,East,Graduate,Self-Employed,139662.121852,620.076082,1.0
4,CUST00005,0,2017-09-30,63903.19,Education,10.0,46.2541,28.000000,Female,South,Graduate,Self-Employed,58780.063998,533.745555,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,CUST00996,0,2022-01-06,36261.91,Car,45.0,13.0978,53.000000,Male,South,Graduate,Salaried,40097.997966,552.440182,1.0
996,CUST00997,0,2017-05-14,18428.59,Business,13.0,41.5562,43.093196,Female,North,Secondary,Salaried,77667.606376,497.628172,0.0
997,CUST00998,0,2017-12-19,17999.50,Car,37.0,16.3216,34.000000,Male,North,Graduate,Salaried,60629.823341,510.063992,3.0
998,CUST00999,0,2022-05-29,18600.19,Home,14.0,37.7159,60.000000,Male,West,Graduate,Self-Employed,88874.494295,533.654370,1.0


In [245]:
# Z-Score method

# we will use df_mice from now on
 
columns = ['age','annual_income','loan_amount','credit_score','transaction_count']

def z_score(df,cols):
    threshold = 3
    
    df_z = df.copy()
    
    for col in cols:
        mean = df_z[col].mean()
        std = df_z[col].std()
        
        z = (df_z[col] - mean) / std
        df_z[col + "z_score"] = z
                
        df_z = df_z[df_z[col + "z_score"].abs() <= threshold].copy()
        df_z = df_z.drop(columns = [col + "z_score"])
    
    return df_z

df_zcore = z_score(df=df_mice, cols = columns)

print('After applying Z-Score')
print(f'Record Removed: {len(df_mice)-len(df_zcore)}')
display(df_zcore)

After applying Z-Score
Record Removed: 57


,customer_id,default_flag,join_date,loan_amount,loan_purpose,transaction_count,spending_ratio,age,gender,region,education_level,employment_type,annual_income,credit_score,repayment_history
0,CUST00001,0,2022-03-04,25700.13,Car,26.0,19.0994,59.000000,Female,South,Graduate,Self-Employed,38384.982865,565.520304,3.0
1,CUST00002,0,2016-04-01,19264.58,Business,2.0,27.1723,49.000000,Female,West,Secondary,Self-Employed,54156.786444,580.911557,2.0
2,CUST00003,0,2015-04-13,23983.44,Education,28.0,41.9300,35.000000,Female,East,Graduate,Self-Employed,88523.013804,621.473062,1.0
3,CUST00004,0,2018-01-31,58439.10,Car,48.0,44.8451,63.000000,Female,East,Graduate,Self-Employed,139662.121852,620.076082,1.0
4,CUST00005,0,2017-09-30,63903.19,Education,10.0,46.2541,28.000000,Female,South,Graduate,Self-Employed,58780.063998,533.745555,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,CUST00996,0,2022-01-06,36261.91,Car,45.0,13.0978,53.000000,Male,South,Graduate,Salaried,40097.997966,552.440182,1.0
996,CUST00997,0,2017-05-14,18428.59,Business,13.0,41.5562,43.093196,Female,North,Secondary,Salaried,77667.606376,497.628172,0.0
997,CUST00998,0,2017-12-19,17999.50,Car,37.0,16.3216,34.000000,Male,North,Graduate,Salaried,60629.823341,510.063992,3.0
998,CUST00999,0,2022-05-29,18600.19,Home,14.0,37.7159,60.000000,Male,West,Graduate,Self-Employed,88874.494295,533.654370,1.0


In [246]:
# outlier handling with IQR method

def iqr(df,cols):
    
    df_iqr = df.copy()
    
    for col in cols:
        Q1 = df[col].quantile(0.25)   
        Q3 = df[col].quantile(0.75)   
        IQR = Q3 - Q1
        
        lower_fence = Q1 - 1.5 * IQR
        upper_fence = Q3 + 1.5 * IQR
        
        outlier_mask = (df[col] < lower_fence) | (df[col] > upper_fence)
        df_iqr = df_iqr[~outlier_mask]
    
    return df_iqr

df_IQR = iqr(df=df_mice, cols=columns)

print('After applying IQR')
print(f'Record Removed: {len(df_mice)-len(df_IQR)}')
display(df_IQR)    

After applying IQR
Record Removed: 158


C:\Users\MAITRAK\AppData\Local\Temp\ipykernel_19840\3929444545.py:16: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_iqr = df_iqr[~outlier_mask]
C:\Users\MAITRAK\AppData\Local\Temp\ipykernel_19840\3929444545.py:16: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_iqr = df_iqr[~outlier_mask]
C:\Users\MAITRAK\AppData\Local\Temp\ipykernel_19840\3929444545.py:16: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_iqr = df_iqr[~outlier_mask]


,customer_id,default_flag,join_date,loan_amount,loan_purpose,transaction_count,spending_ratio,age,gender,region,education_level,employment_type,annual_income,credit_score,repayment_history
0,CUST00001,0,2022-03-04,25700.13,Car,26.0,19.0994,59.000000,Female,South,Graduate,Self-Employed,38384.982865,565.520304,3.0
1,CUST00002,0,2016-04-01,19264.58,Business,2.0,27.1723,49.000000,Female,West,Secondary,Self-Employed,54156.786444,580.911557,2.0
2,CUST00003,0,2015-04-13,23983.44,Education,28.0,41.9300,35.000000,Female,East,Graduate,Self-Employed,88523.013804,621.473062,1.0
3,CUST00004,0,2018-01-31,58439.10,Car,48.0,44.8451,63.000000,Female,East,Graduate,Self-Employed,139662.121852,620.076082,1.0
4,CUST00005,0,2017-09-30,63903.19,Education,10.0,46.2541,28.000000,Female,South,Graduate,Self-Employed,58780.063998,533.745555,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,CUST00996,0,2022-01-06,36261.91,Car,45.0,13.0978,53.000000,Male,South,Graduate,Salaried,40097.997966,552.440182,1.0
996,CUST00997,0,2017-05-14,18428.59,Business,13.0,41.5562,43.093196,Female,North,Secondary,Salaried,77667.606376,497.628172,0.0
997,CUST00998,0,2017-12-19,17999.50,Car,37.0,16.3216,34.000000,Male,North,Graduate,Salaried,60629.823341,510.063992,3.0
998,CUST00999,0,2022-05-29,18600.19,Home,14.0,37.7159,60.000000,Male,West,Graduate,Self-Employed,88874.494295,533.654370,1.0


In [247]:
# Percentile

def percentile(df, cols, lower, upper):
    df_perc = df.copy()

    for col in cols:
        lower_bound = df_perc[col].quantile(lower)
        upper_bound = df_perc[col].quantile(upper)

        outlier_mask = (df_perc[col] < lower_bound) | (df_perc[col] > upper_bound)

        df_perc = df_perc[~outlier_mask]

    return df_perc

df_percentile = percentile(df=df_mice, cols=columns, lower=.05, upper=.95)

print('After applying percentile method')
print(f'Record Removed: {len(df_mice)-len(df_percentile)}')
display(df_percentile)

After applying percentile method
Record Removed: 391


,customer_id,default_flag,join_date,loan_amount,loan_purpose,transaction_count,spending_ratio,age,gender,region,education_level,employment_type,annual_income,credit_score,repayment_history
2,CUST00003,0,2015-04-13,23983.44,Education,28.0,41.9300,35.000000,Female,East,Graduate,Self-Employed,88523.013804,621.473062,1.0
4,CUST00005,0,2017-09-30,63903.19,Education,10.0,46.2541,28.000000,Female,South,Graduate,Self-Employed,58780.063998,533.745555,2.0
6,CUST00007,0,2016-07-25,17727.11,Home,37.0,17.6691,59.000000,Female,North,Primary,Salaried,47573.714010,548.864517,0.0
7,CUST00008,0,2016-02-24,20571.90,Car,39.0,31.2673,39.000000,Male,East,Graduate,Unemployed,177943.212904,587.497893,0.0
8,CUST00009,0,2022-08-03,34501.93,Car,21.0,42.1541,43.000000,Male,South,Graduate,Self-Employed,276619.631043,654.941926,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,CUST00996,0,2022-01-06,36261.91,Car,45.0,13.0978,53.000000,Male,South,Graduate,Salaried,40097.997966,552.440182,1.0
996,CUST00997,0,2017-05-14,18428.59,Business,13.0,41.5562,43.093196,Female,North,Secondary,Salaried,77667.606376,497.628172,0.0
997,CUST00998,0,2017-12-19,17999.50,Car,37.0,16.3216,34.000000,Male,North,Graduate,Salaried,60629.823341,510.063992,3.0
998,CUST00999,0,2022-05-29,18600.19,Home,14.0,37.7159,60.000000,Male,West,Graduate,Self-Employed,88874.494295,533.654370,1.0


In [248]:
def winsorization(df, cols, lower, upper):
    df_win = df.copy()

    for col in cols:
        lower_win = df_win[col].quantile(lower)
        upper_win = df_win[col].quantile(upper)

        df_win[col] = df_win[col].clip(lower=lower_win, upper=upper_win)

    return df_win

df_winsorizarion = winsorization(df=df_mice,cols=columns, lower =.05,upper=.95)
print('After applying winsorization method')
print(f'Record Removed: {len(df_mice)-len(df_winsorizarion)}')
display(df_winsorizarion)

After applying winsorization method
Record Removed: 0


,customer_id,default_flag,join_date,loan_amount,loan_purpose,transaction_count,spending_ratio,age,gender,region,education_level,employment_type,annual_income,credit_score,repayment_history
0,CUST00001,0,2022-03-04,25700.13,Car,26.0,19.0994,59.000000,Female,South,Graduate,Self-Employed,39415.322948,565.520304,3.0
1,CUST00002,0,2016-04-01,19264.58,Business,3.0,27.1723,49.000000,Female,West,Secondary,Self-Employed,54156.786444,580.911557,2.0
2,CUST00003,0,2015-04-13,23983.44,Education,28.0,41.9300,35.000000,Female,East,Graduate,Self-Employed,88523.013804,621.473062,1.0
3,CUST00004,0,2018-01-31,58439.10,Car,47.0,44.8451,62.000000,Female,East,Graduate,Self-Employed,139662.121852,620.076082,1.0
4,CUST00005,0,2017-09-30,63903.19,Education,10.0,46.2541,28.000000,Female,South,Graduate,Self-Employed,58780.063998,533.745555,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,CUST00996,0,2022-01-06,36261.91,Car,45.0,13.0978,53.000000,Male,South,Graduate,Salaried,40097.997966,552.440182,1.0
996,CUST00997,0,2017-05-14,18428.59,Business,13.0,41.5562,43.093196,Female,North,Secondary,Salaried,77667.606376,497.628172,0.0
997,CUST00998,0,2017-12-19,17999.50,Car,37.0,16.3216,34.000000,Male,North,Graduate,Salaried,60629.823341,510.063992,3.0
998,CUST00999,0,2022-05-29,18600.19,Home,14.0,37.7159,60.000000,Male,West,Graduate,Self-Employed,88874.494295,533.654370,1.0


## Part-E Feature Engineering

In [249]:

# handling variable types
# As there are no mixed data that will not be necessary
# From here on out we will use df_IQR as it has removed outliers while preserving data
# Date & Time Variables

df_IQR["join_date"] = pd.to_datetime(df_IQR["join_date"])

df_IQR["year"] = df_IQR["join_date"].dt.year
df_IQR["month"] = df_IQR["join_date"].dt.month
df_IQR["day"] = df_IQR["join_date"].dt.day
df_IQR["weekday"] = df_IQR["join_date"].dt.dayofweek

print(df_IQR[["join_date","year","month","day","weekday"]])

df_IQR.drop(["year","month","day","weekday"], axis=1)

     join_date  year  month  day  weekday
0   2022-03-04  2022      3    4        4
1   2016-04-01  2016      4    1        4
2   2015-04-13  2015      4   13        0
3   2018-01-31  2018      1   31        2
4   2017-09-30  2017      9   30        5
..         ...   ...    ...  ...      ...
995 2022-01-06  2022      1    6        3
996 2017-05-14  2017      5   14        6
997 2017-12-19  2017     12   19        1
998 2022-05-29  2022      5   29        6
999 2015-11-29  2015     11   29        6

[842 rows x 5 columns]


,customer_id,default_flag,join_date,loan_amount,loan_purpose,transaction_count,spending_ratio,age,gender,region,education_level,employment_type,annual_income,credit_score,repayment_history
0,CUST00001,0,2022-03-04,25700.13,Car,26.0,19.0994,59.000000,Female,South,Graduate,Self-Employed,38384.982865,565.520304,3.0
1,CUST00002,0,2016-04-01,19264.58,Business,2.0,27.1723,49.000000,Female,West,Secondary,Self-Employed,54156.786444,580.911557,2.0
2,CUST00003,0,2015-04-13,23983.44,Education,28.0,41.9300,35.000000,Female,East,Graduate,Self-Employed,88523.013804,621.473062,1.0
3,CUST00004,0,2018-01-31,58439.10,Car,48.0,44.8451,63.000000,Female,East,Graduate,Self-Employed,139662.121852,620.076082,1.0
4,CUST00005,0,2017-09-30,63903.19,Education,10.0,46.2541,28.000000,Female,South,Graduate,Self-Employed,58780.063998,533.745555,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,CUST00996,0,2022-01-06,36261.91,Car,45.0,13.0978,53.000000,Male,South,Graduate,Salaried,40097.997966,552.440182,1.0
996,CUST00997,0,2017-05-14,18428.59,Business,13.0,41.5562,43.093196,Female,North,Secondary,Salaried,77667.606376,497.628172,0.0
997,CUST00998,0,2017-12-19,17999.50,Car,37.0,16.3216,34.000000,Male,North,Graduate,Salaried,60629.823341,510.063992,3.0
998,CUST00999,0,2022-05-29,18600.19,Home,14.0,37.7159,60.000000,Male,West,Graduate,Self-Employed,88874.494295,533.654370,1.0


In [250]:
# Encoding categorical Variables
# Ordinal encoding

print(df_IQR["education_level"].value_counts())

education_order = [["Primary", "Secondary", "Graduate", "Post-Graduate"]]  

enc = OrdinalEncoder(categories=education_order)

df_IQR["education_level_encoded"] = enc.fit_transform(df_IQR[["education_level"]]) 

print(df_IQR[["education_level", "education_level_encoded"]])

education_level
Graduate         394
Secondary        200
Post-Graduate    161
Primary           87
Name: count, dtype: int64
    education_level  education_level_encoded
0          Graduate                      2.0
1         Secondary                      1.0
2          Graduate                      2.0
3          Graduate                      2.0
4          Graduate                      2.0
..              ...                      ...
995        Graduate                      2.0
996       Secondary                      1.0
997        Graduate                      2.0
998        Graduate                      2.0
999        Graduate                      2.0

[842 rows x 2 columns]


In [251]:
# Label encoding

le = LabelEncoder()

df_IQR["employment_type_encoded"] = le.fit_transform(df_IQR["employment_type"])

df_IQR["gender_encoded"] = le.fit_transform(df_IQR["gender"])

display(df_IQR[["employment_type","employment_type_encoded","gender","gender_encoded"]])


,employment_type,employment_type_encoded,gender,gender_encoded
0,Self-Employed,1,Female,0
1,Self-Employed,1,Female,0
2,Self-Employed,1,Female,0
3,Self-Employed,1,Female,0
4,Self-Employed,1,Female,0
...,...,...,...,...
995,Salaried,0,Male,1
996,Salaried,0,Female,0
997,Salaried,0,Male,1
998,Self-Employed,1,Male,1


In [252]:
# one-Hot encoding

ohe = OneHotEncoder(drop = "first",sparse_output=False, handle_unknown="ignore")
cols = ["region","loan_purpose"]
encoded_arr = ohe.fit_transform(df_IQR[cols])
feature_names = ohe.get_feature_names_out(cols)

df_ohe = pd.DataFrame(encoded_arr, columns=feature_names, index=df_IQR.index)

df_IQR = pd.concat([df_IQR, df_ohe], axis=1)

print("New columns added:", list(feature_names))
display(df_IQR[["region", "loan_purpose"] + list(feature_names)])



New columns added: ['region_North', 'region_South', 'region_West', 'loan_purpose_Car', 'loan_purpose_Education', 'loan_purpose_Home', 'loan_purpose_Other']


,region,loan_purpose,region_North,region_South,region_West,loan_purpose_Car,loan_purpose_Education,loan_purpose_Home,loan_purpose_Other
0,South,Car,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1,West,Business,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,East,Education,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,East,Car,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,South,Education,0.0,1.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
995,South,Car,0.0,1.0,0.0,1.0,0.0,0.0,0.0
996,North,Business,1.0,0.0,0.0,0.0,0.0,0.0,0.0
997,North,Car,1.0,0.0,0.0,1.0,0.0,0.0,0.0
998,West,Home,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [253]:
# Encoding Numerical Variables
